In [ ]:
import numpy as np
import json

X_train = np.load('../data/processed/X_train.npy')
X_test  = np.load('../data/processed/X_test.npy')
y_train = np.load('../data/processed/y_train.npy')
y_test  = np.load('../data/processed/y_test.npy')

with open('../data/label_encoding.json') as f:
    mapping = json.load(f)
class_names = list(mapping.keys())

print("X_train :", X_train.shape)
print("Classes :", class_names)

In [ ]:
import numpy as np
import json
from sklearn.ensemble import RandomForestClassifier
import joblib, time
from sklearn.utils import resample

X_train = np.load('../data/processed/X_train.npy')
X_test  = np.load('../data/processed/X_test.npy')
y_train = np.load('../data/processed/y_train.npy')
y_test  = np.load('../data/processed/y_test.npy')

print("Shape original :", X_train.shape)

X_sample, y_sample = resample(
    X_train, y_train,
    n_samples=200000,
    random_state=42
)
print("Shape échantillon :", X_sample.shape)

t0 = time.time()
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_split=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_sample, y_sample)
print(f"✅ Entraîné en {time.time()-t0:.1f}s")

joblib.dump(rf, '../models/random_forest_model.pkl')
print("✅ Modèle sauvegardé !")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns, matplotlib.pyplot as plt

y_pred = rf.predict(X_test)

print(classification_report(y_test, y_pred, target_names=class_names))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title("Random Forest — Confusion Matrix")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../data/rf_confusion_matrix.png')
plt.show()

In [ ]:
# Test sur un exemple — simule ce que recevra P4
example = X_test[0:1]
pred_class    = rf.predict(example)[0]
pred_proba    = rf.predict_proba(example)[0]
confidence    = pred_proba.max()
label_name    = class_names[pred_class]

print(f"Prédiction    : {label_name}")
print(f"Confiance     : {confidence:.2%}")
print(f"Toutes probas : { {class_names[i]: f'{p:.2%}' for i,p in enumerate(pred_proba)} }")